# SHIPP Stage 13 — Marketplace Analytics

**Owner:** Raneem  
**Priority:** P0  
**Current gate:** 13.0 — Analytics CDF source readiness

## Goal

Verify that the two operational Agent-write tables:

- `shipp.saved_items`
- `shipp.agent_activity`

are arriving in Bronze through Lakebase CDF before we build
`gold_marketplace_metrics`.

**This gate is read-only.** It does not create or modify analytics data.

### PASS condition

Both expected Bronze CDF tables must:

1. exist,
2. contain at least one row,
3. expose the SHIPP Lakebase CDF ordering metadata:
   - `_pg_change_type`
   - `_pg_lsn`
   - `_sort_by`


In [0]:
%run ../common/shipp_silver_lib

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.0A
# Verify canonical Analytics table registration.
# READ ONLY.
# -------------------------------------------------------------------

EXPECTED_SAVED_ITEMS_HISTORY = (
    f"{CATALOG}.{BRONZE_SCHEMA}.lb_saved_items_history"
)

EXPECTED_AGENT_ACTIVITY_HISTORY = (
    f"{CATALOG}.{BRONZE_SCHEMA}.lb_agent_activity_history"
)

EXPECTED_GOLD_METRICS = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_marketplace_metrics"
)


assert BRONZE_SAVED_ITEMS_HISTORY == EXPECTED_SAVED_ITEMS_HISTORY, (
    "CONFIG ERROR: BRONZE_SAVED_ITEMS_HISTORY is incorrect.\n"
    f"Expected: {EXPECTED_SAVED_ITEMS_HISTORY}\n"
    f"Actual:   {BRONZE_SAVED_ITEMS_HISTORY}"
)


assert BRONZE_AGENT_ACTIVITY_HISTORY == EXPECTED_AGENT_ACTIVITY_HISTORY, (
    "CONFIG ERROR: BRONZE_AGENT_ACTIVITY_HISTORY is incorrect.\n"
    f"Expected: {EXPECTED_AGENT_ACTIVITY_HISTORY}\n"
    f"Actual:   {BRONZE_AGENT_ACTIVITY_HISTORY}"
)


assert GOLD_MARKETPLACE_METRICS == EXPECTED_GOLD_METRICS, (
    "CONFIG ERROR: GOLD_MARKETPLACE_METRICS is incorrect.\n"
    f"Expected: {EXPECTED_GOLD_METRICS}\n"
    f"Actual:   {GOLD_MARKETPLACE_METRICS}"
)


print("PASS — Stage 13 canonical table names are registered.")
print()
print("Saved Items CDF :", BRONZE_SAVED_ITEMS_HISTORY)
print("Agent Activity  :", BRONZE_AGENT_ACTIVITY_HISTORY)
print("Gold target     :", GOLD_MARKETPLACE_METRICS)

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.0B
# Verify real Lakebase CDF sources.
#
# READ ONLY.
#
# Required SHIPP Lakebase CDC metadata:
#   _pg_change_type
#   _pg_lsn
#   _sort_by
# -------------------------------------------------------------------

ANALYTICS_CDF_SOURCES = {
    "saved_items": BRONZE_SAVED_ITEMS_HISTORY,
    "agent_activity": BRONZE_AGENT_ACTIVITY_HISTORY,
}

REQUIRED_CDF_COLUMNS = {
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
}

NUMERIC_SPARK_TYPES = {
    "byte",
    "short",
    "integer",
    "long",
    "float",
    "double",
    "decimal",
}


print(
    "=== STAGE 13 — ANALYTICS CDF SOURCE READINESS ==="
)


source_results = {}


for source_name, table_name in ANALYTICS_CDF_SOURCES.items():

    print()
    print("--------------------------------------------------")
    print("Source:", source_name)
    print("Table :", table_name)
    print("--------------------------------------------------")

    exists = table_exists(table_name)

    print("Exists:", exists)

    source_results[source_name] = {
        "table": table_name,
        "exists": exists,
        "rows": None,
        "missing_columns": None,
        "change_types": None,
    }


    # ---------------------------------------------------------------
    # Table must actually exist.
    # ---------------------------------------------------------------

    if not exists:

        print(
            "BLOCKED — expected Lakebase CDF "
            "history table does not exist."
        )

        continue


    # ---------------------------------------------------------------
    # Read the real Bronze CDF table.
    # ---------------------------------------------------------------

    df = spark.table(
        table_name
    )

    row_count = df.count()

    print("Rows:", row_count)

    print()
    print("Schema:")

    df.printSchema()


    # ---------------------------------------------------------------
    # Required CDC metadata
    # ---------------------------------------------------------------

    missing_columns = sorted(
        REQUIRED_CDF_COLUMNS
        - set(df.columns)
    )

    source_results[source_name][
        "rows"
    ] = row_count

    source_results[source_name][
        "missing_columns"
    ] = missing_columns


    assert not missing_columns, (
        f"BLOCKED: {table_name} is missing "
        f"required CDF columns: "
        f"{missing_columns}"
    )


    # ---------------------------------------------------------------
    # We need real events, not just an empty table.
    # ---------------------------------------------------------------

    assert row_count > 0, (
        f"BLOCKED: {table_name} exists "
        "but contains 0 rows. "
        "Stage 13 needs real Agent-write "
        "CDF evidence."
    )


    # ---------------------------------------------------------------
    # Validate ordering metadata types.
    #
    # Current-state reconstruction depends on:
    #   _pg_lsn DESC
    #   _sort_by DESC
    # ---------------------------------------------------------------

    lsn_type = (
        df.schema["_pg_lsn"]
        .dataType
    )

    sort_type = (
        df.schema["_sort_by"]
        .dataType
    )


    print()
    print("LSN TYPE  =", lsn_type)
    print("SORT TYPE =", sort_type)


    assert lsn_type.typeName() in NUMERIC_SPARK_TYPES, (
        f"BLOCKED: {table_name}._pg_lsn "
        f"is not numeric: {lsn_type}"
    )


    assert sort_type.typeName() in NUMERIC_SPARK_TYPES, (
        f"BLOCKED: {table_name}._sort_by "
        f"is not numeric: {sort_type}"
    )


    # ---------------------------------------------------------------
    # Show actual change types present.
    # ---------------------------------------------------------------

    change_types = [
        row["_pg_change_type"]
        for row in (
            df
            .select("_pg_change_type")
            .distinct()
            .orderBy("_pg_change_type")
            .collect()
        )
    ]


    source_results[source_name][
        "change_types"
    ] = change_types


    print()
    print(
        "Change types:",
        change_types,
    )


    # ---------------------------------------------------------------
    # Show newest events for runtime evidence.
    # ---------------------------------------------------------------

    print()
    print("Latest CDF rows:")

    display(
        df
        .orderBy(
            F.col("_pg_lsn").desc(),
            F.col("_sort_by").desc(),
        )
        .limit(10)
    )


# -------------------------------------------------------------------
# Final source existence gate.
# -------------------------------------------------------------------

missing_sources = [
    details["table"]
    for details
    in source_results.values()
    if not details["exists"]
]


assert not missing_sources, (
    "STAGE 13 BLOCKED — "
    "missing CDF source(s): "
    + ", ".join(missing_sources)
)


print()
print(
    "PASS Gate 13.0 — saved_items and "
    "agent_activity CDF histories exist, "
    "contain real rows, and expose valid "
    "CDC ordering metadata."
)

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.0 Summary
# -------------------------------------------------------------------

gate_13_0_summary = {
    "stage": 13,
    "gate": "13.0_cdf_source_readiness",

    "saved_items_history": {
        "table": BRONZE_SAVED_ITEMS_HISTORY,
        "exists": source_results["saved_items"]["exists"],
        "rows": source_results["saved_items"]["rows"],
        "change_types": source_results["saved_items"]["change_types"],
    },

    "agent_activity_history": {
        "table": BRONZE_AGENT_ACTIVITY_HISTORY,
        "exists": source_results["agent_activity"]["exists"],
        "rows": source_results["agent_activity"]["rows"],
        "change_types": source_results["agent_activity"]["change_types"],
    },

    "target": GOLD_MARKETPLACE_METRICS,
    "status": "PASS",
}

print(
    json.dumps(
        gate_13_0_summary,
        indent=2,
        default=str,
    )
)

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.1
# Reconstruct trusted current state from Analytics CDF histories.
#
# READ ONLY.
#
# CDC rule:
#   1. rank insert / update_postimage / delete
#   2. _pg_lsn DESC
#   3. _sort_by DESC
#   4. select newest event per business key
#   5. newest DELETE = entity excluded from current state
#
# IMPORTANT:
# Never filter DELETE before ranking.
# -------------------------------------------------------------------

print("=== STAGE 13 — TRUSTED ANALYTICS CURRENT STATE ===")


# ================================================================
# Saved Items
# Business key: saved_item_id
# ================================================================

saved_items_bronze = spark.table(
    BRONZE_SAVED_ITEMS_HISTORY
)

(
    saved_items_latest,
    saved_items_current,
    saved_items_deleted,
) = reconstruct_current_state(
    bronze_df=saved_items_bronze,
    key="saved_item_id",
)


saved_items_bronze_count = (
    saved_items_bronze.count()
)

saved_items_latest_count = (
    saved_items_latest.count()
)

saved_items_current_count = (
    saved_items_current.count()
)

saved_items_deleted_count = (
    saved_items_deleted.count()
)

saved_items_duplicate_current = (
    saved_items_current
    .groupBy("saved_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


print()
print("--- SAVED ITEMS ---")
print(
    "Bronze CDF rows       :",
    saved_items_bronze_count,
)
print(
    "Latest business keys  :",
    saved_items_latest_count,
)
print(
    "Current saved items   :",
    saved_items_current_count,
)
print(
    "Latest DELETE keys    :",
    saved_items_deleted_count,
)
print(
    "Duplicate current keys:",
    saved_items_duplicate_current,
)


assert saved_items_duplicate_current == 0, (
    "FAILED: duplicate saved_item_id "
    "after current-state reconstruction."
)


print()
print("Current Saved Items:")

display(
    saved_items_current
    .select(
        "saved_item_id",
        "user_id",
        "request_id",
        "listing_id",
        "status",
        "saved_at",
        "_pg_lsn",
        "_sort_by",
        "_pg_change_type",
    )
    .orderBy("saved_item_id")
)


# ================================================================
# Agent Activity
# Business key: activity_id
# ================================================================

agent_activity_bronze = spark.table(
    BRONZE_AGENT_ACTIVITY_HISTORY
)

(
    agent_activity_latest,
    agent_activity_current,
    agent_activity_deleted,
) = reconstruct_current_state(
    bronze_df=agent_activity_bronze,
    key="activity_id",
)


agent_activity_bronze_count = (
    agent_activity_bronze.count()
)

agent_activity_latest_count = (
    agent_activity_latest.count()
)

agent_activity_current_count = (
    agent_activity_current.count()
)

agent_activity_deleted_count = (
    agent_activity_deleted.count()
)

agent_activity_duplicate_current = (
    agent_activity_current
    .groupBy("activity_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


print()
print("--- AGENT ACTIVITY ---")
print(
    "Bronze CDF rows       :",
    agent_activity_bronze_count,
)
print(
    "Latest business keys  :",
    agent_activity_latest_count,
)
print(
    "Current activity rows :",
    agent_activity_current_count,
)
print(
    "Latest DELETE keys    :",
    agent_activity_deleted_count,
)
print(
    "Duplicate current keys:",
    agent_activity_duplicate_current,
)


assert agent_activity_duplicate_current == 0, (
    "FAILED: duplicate activity_id "
    "after current-state reconstruction."
)


print()
print("Agent Activity by status:")

display(
    agent_activity_current
    .groupBy(
        "tool_name",
        "action_status",
    )
    .count()
    .orderBy(
        "tool_name",
        "action_status",
    )
)


print()
print(
    "PASS Gate 13.1 — trusted Saved Item and "
    "Agent Activity current states reconstructed "
    "with delete-aware CDC semantics."
)